# Close All Subscriber Accounts

This notebook:
1. Generates an OAuth2 access token (password grant)
2. Fetches all subscriber accounts for a partner via `GET /rest/SubscriberService/v1/subscribers`
3. Iterates through each account and closes it via `DELETE /rest/SubscriberService/v1/subscribers/{accountNumber}`

**Safety:** `DRY_RUN = True` by default. The notebook will list what *would* be closed without actually closing anything. Flip it to `False` once you've reviewed the list.

## 1. Configuration

Set your host, OAuth credentials, and the dry-run flag here.

In [14]:
import os
import time
import requests
from requests.exceptions import RequestException
from sqlalchemy import create_engine
from datetime import datetime, timedelta
import pandas as pd
import logging

# --- API host ---
HOST = os.getenv("API_HOST", "https://sandbox-sg.onebillsoftware.com")

# --- OAuth2 password-grant credentials ---
TOKEN_URL = f"{HOST}/oauth/token"   # <- adjust path if your token endpoint differs
OAUTH_USERNAME = os.getenv("API_USERNAME")
OAUTH_PASSWORD = os.getenv("API_PASSWORD")
OAUTH_CLIENT_ID = os.getenv("CLIENT_ID")
OAUTH_CLIENT_SECRET = os.getenv("CLIENT_SECRET")  # leave blank if not required           

# --- Behaviour ---
DRY_RUN = False              # Set to False to actually close accounts
REQUEST_TIMEOUT = 30          # seconds
MAX_WORKERS = 10              # parallel close requests. Bump to 20-30 if API tolerates it.

# Optional: only close accounts matching this filter. Leave as None to close ALL.
# Example: lambda s: s["accountType"] == 1001
ACCOUNT_FILTER = None

SUBSCRIBERS_URL = f"{HOST}/rest/SubscriberService/v1/subscribers"

BI_DATASTORE_URL = (
    f"mysql+mysqlconnector://{os.environ['DB_USERNAME']}:{os.environ['DB_PASSWORD']}"
    f"@{os.environ['DB_HOST']}/bi_datastore"
)

# --- Logging ---
log_filename = f'migration_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler(log_filename), logging.StreamHandler()],
)
logger = logging.getLogger(__name__)

print(f"Host:    {HOST}")
print(f"Dry run: {DRY_RUN}")

Host:    https://sandbox-sg.onebillsoftware.com
Dry run: False


## 2. Generate access token

Calls the token endpoint with `grant_type=password`. The token is cached so the rest of the notebook reuses it.

In [9]:
def fetch_access_token():
    """Get an OAuth2 access token using the password grant."""
    data = {
        "grant_type": "password",
        "username": OAUTH_USERNAME,
        "password": OAUTH_PASSWORD,
        "client_id": OAUTH_CLIENT_ID,
    }
    if OAUTH_CLIENT_SECRET:
        data["client_secret"] = OAUTH_CLIENT_SECRET

    headers = {
        "Content-Type": "application/x-www-form-urlencoded",
        "Accept": "application/json",
    }

    resp = requests.post(TOKEN_URL, data=data, headers=headers, timeout=REQUEST_TIMEOUT)
    if resp.status_code != 200:
        raise RuntimeError(
            f"Token request failed: HTTP {resp.status_code} - {resp.text[:300]}"
        )
    payload = resp.json()
    token = payload.get("access_token")
    if not token:
        raise RuntimeError(f"No access_token in response: {payload}")

    expires_in = payload.get("expires_in")
    token_type = payload.get("token_type", "Bearer")
    if expires_in:
        print(f"Got {token_type} token. Expires in: {expires_in}s")
    else:
        print(f"Got {token_type} token.")
    return token, token_type

ACCESS_TOKEN, TOKEN_TYPE = fetch_access_token()

HEADERS = {
    "Authorization": f"{TOKEN_TYPE} {ACCESS_TOKEN}",
    "Accept": "application/json",
    "Content-Type": "application/json",
    "proxy_accountNumber": os.environ["DELETION_PROXY_ACCOUNT_NUMBER"]
}

Got bearer token. Expires in: 957s


## 3. Fetch all subscribers

In [16]:
ACCOUNT_QUERY = """
SELECT 
	*
FROM 
	bi_datastore.billing_account
WHERE 
	`_DataSource` = 'vBill'
AND 
	`Status` = 'DEACTIVATED'
""".strip()

engine = create_engine(BI_DATASTORE_URL)
df_accounts = pd.read_sql(ACCOUNT_QUERY, con=engine)

# AccountCode as string for consistent dict lookups against Dataverse-side strings
df_accounts["AccountCode"] = df_accounts["AccountCode"].astype(str)

logger.info(f"Loaded {len(df_accounts):,} accounts from MySQL")
df_accounts.head()

2026-05-18 15:14:52,031 [INFO] Loaded 11,065 accounts from MySQL


,_rowid,_rowmodified,_sourceid,_DataSource,AccountCode,AccountName,AlternateAccountCode,BillToAccountCode,BillToAccountName,CreatedDate,...,_temporary_crmonly_addr1,_temporary_crmonly_addr2,_temporary_crmonly_suburb,_temporary_crmonly_city,_temporary_crmonly_postcode,_temporary_crmonly_countryiso,_temporary_crmonly_phone_home,_temporary_crmonly_phone_work,_temporary_crmonly_mobile,_temporary_crmonly_dob
0,181219,2025-05-24 04:30:21,6,vBill,1000000008,Inomial,None,None,None,2005-12-08,...,,None,None,None,None,None,None,None,None,None
1,104399,2025-05-24 04:32:58,18,vBill,1000000024,"Sales, Cash",None,None,None,2005-12-22,...,,None,None,None,None,None,None,None,None,None
2,251068597,2022-06-21 18:59:39,219316,vBill,1099978402,Jay Gandhi (Operator),None,None,None,2021-10-27,...,14 Fenwick Crescent,None,Wallaceville,Upper Hutt,5018,NZ,+642040070004,None,+642040070004,1996-06-05
3,132819,2021-04-04 14:19:48,10319,vBill,20115569,The Total Saver Limited (In Liquidation) (ICMS),None,None,None,2018-06-14,...,253 Fitzgerald Road,None,Drury,,2577,None,None,+6499730974,None,None
4,156379,2021-04-04 14:20:00,10320,vBill,20241342,Pan Pacific Travel Corporation Limited,None,None,None,2018-06-14,...,"Level 1, 333 Remuera Road",None,Remuera,,,None,None,+6495209193,,None


## 4. Preview the accounts to be closed

In [18]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

session = requests.Session()
session.headers.update(HEADERS)

def close_account(account_number: str):
    url = f"{HOST}/rest/SubscriberService/v1/subscribers/{account_number}"
    try:
        resp = session.delete(url, timeout=REQUEST_TIMEOUT)
        if 200 <= resp.status_code < 300:
            return True, f"HTTP {resp.status_code}"
        return False, f"HTTP {resp.status_code}: {resp.text[:200]}"
    except RequestException as e:
        return False, f"Exception: {e}"

results = {"closed": [], "failed": [], "skipped": []}
overall_start = time.time()
batch_num = 0
MAX_BATCHES = 100   # safety cap so we can't loop forever if the API misbehaves

while batch_num < MAX_BATCHES:
    batch_num += 1
    print(f"\n=== Batch {batch_num} ===")
    batch, total_count = get_subscribers_batch()

    # Apply optional filter
    if ACCOUNT_FILTER is not None:
        batch = [s for s in batch if ACCOUNT_FILTER(s)]

    # Drop any without an accountNumber
    valid = []
    for sub in batch:
        if sub.get("accountNumber"):
            valid.append(sub)
        else:
            results["skipped"].append(sub)

    if not valid:
        print("No more accounts to close. Done.")
        break

    print(f"Closing {len(valid)} accounts with {MAX_WORKERS} workers...")

    if DRY_RUN:
        for sub in valid:
            print(f"  DRY-RUN would close {sub.get('accountNumber')} ({sub.get('accountName','')})")
        print("Dry run: stopping after first batch so we don't loop forever.")
        break

    progress = {"done": 0}
    progress_lock = Lock()
    total = len(valid)

    def worker(sub):
        acct = sub.get("accountNumber")
        name = sub.get("accountName", "")
        ok, detail = close_account(acct)
        with progress_lock:
            progress["done"] += 1
            status = "CLOSED" if ok else "FAILED"
            print(f"  [{progress['done']}/{total}] {status}  {acct} ({name}) - {detail}")
        return acct, ok, detail

    closed_in_batch = 0
    failed_in_batch = 0
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(worker, sub) for sub in valid]
        for fut in as_completed(futures):
            acct, ok, detail = fut.result()
            if ok:
                results["closed"].append(acct)
                closed_in_batch += 1
            else:
                results["failed"].append({"accountNumber": acct, "error": detail})
                failed_in_batch += 1

    print(f"Batch {batch_num} done: closed {closed_in_batch}, failed {failed_in_batch}")

    # Safety: if every account in the batch failed, stop — we'd loop forever otherwise
    if closed_in_batch == 0 and failed_in_batch > 0:
        print("\nALL accounts in this batch failed to close. Stopping to avoid an infinite loop.")
        print("Check the failures above (auth? permissions? wrong HTTP verb?) and re-run.")
        break

elapsed = time.time() - overall_start
print(f"\n=== Overall: {elapsed:.1f}s, closed {len(results['closed'])}, "
      f"failed {len(results['failed'])}, skipped {len(results['skipped'])} ===")


=== Batch 1 ===
  Fetched batch of 1000 (server reports totalCount=48204)
Closing 1000 accounts with 10 workers...
  [1/1000] CLOSED  SR31903 (Kylie Glenn) - HTTP 200
  [2/1000] CLOSED  SR31822 (Lee Miller) - HTTP 200
  [3/1000] CLOSED  SR31803 (Helen Woodhouse) - HTTP 200
  [4/1000] CLOSED  SR31928 (Logan Newport) - HTTP 200
  [5/1000] CLOSED  SR31824 (Accounts Payable) - HTTP 200
  [6/1000] CLOSED  SR31823 (Josh Yeats) - HTTP 200
  [7/1000] CLOSED  SR31825 (Benjamin Salt) - HTTP 200
  [8/1000] CLOSED  SR31820 (Michael Provost) - HTTP 200
  [9/1000] CLOSED  SR31821 (Jonathan Crawford) - HTTP 200
  [10/1000] CLOSED  SR31902 (Anna Milroy) - HTTP 200
  [11/1000] CLOSED  SR31934 (Fred Wadia) - HTTP 200
  [12/1000] CLOSED  SR31826 (Carol Oakes) - HTTP 200
  [13/1000] CLOSED  SR31827 (Seeby Woodhouse) - HTTP 200
  [14/1000] CLOSED  SR31828 (Barry Owen Mclean) - HTTP 200
  [15/1000] CLOSED  SR31935 (Owen Moore) - HTTP 200
  [16/1000] CLOSED  SR31905 (Nathan Kennedy) - HTTP 200
  [17/1000] C

## 6. Summary of failures (if any)

In [19]:
if not DRY_RUN and results["failed"]:
    print("The following accounts failed to close:\n")
    for f in results["failed"]:
        print(f"  {f['accountNumber']}: {f['error']}")
else:
    print("No failures to report.")

No failures to report.


## 7. (Optional) Verify by re-fetching

In [20]:
if not DRY_RUN:
    remaining = get_subscribers()
    print(f"\nRemaining subscribers: {len(remaining)}")
    for s in remaining:
        print(f"  {s.get('accountNumber')} - {s.get('accountName')}")
else:
    print("Dry run was on; skipping verification fetch.")

  Page offset=0: got 0 (running total 0 of 0)
API reported totalCount: 0, fetched: 0

Remaining subscribers: 0
